# 🤗 x 🦾: Training SmolVLA on the SO-101 MuJoCo dataset

Adapted from HuggingFace's official [Train SmolVLA with LeRobot](https://github.com/huggingface/notebooks/blob/main/lerobot/training-smolvla.ipynb) notebook for the `so101_mujoco_sim2real` project's own recorded dataset.

**Two corrections vs. the stock template**, worth knowing before you run this:

1. **`--policy.path=lerobot/smolvla_base` instead of `--policy.type=smolvla`.** The official notebook's training cell uses `--policy.type=smolvla` with no `--policy.path` -- checked against `lerobot`'s own `TrainPipelineConfig` source: without `--policy.path` (or `--policy.pretrained_path` on resume), the policy is constructed fresh from CLI args and defaults, which does **not** load `lerobot/smolvla_base`'s pretrained weights (`load_vlm_weights` defaults to `False`). That means the stock cell would train the whole 450M-parameter model -- vision-language backbone included -- completely from scratch on just a few dozen episodes, which is very unlikely to work. LeRobot's own docs page for SmolVLA (`docs/source/smolvla.mdx`) uses `--policy.path=lerobot/smolvla_base` for exactly this reason; this notebook follows the docs, not the notebook template's literal command.
2. **`--policy.empty_cameras=1` and `--rename_map=...` added.** `smolvla_base` was pretrained expecting 3 camera slots (`camera1`/`camera2`/`camera3`); our dataset has 2 (`observation.images.front`, `observation.images.wrist` -- confirmed directly against the pushed dataset's own metadata before writing this notebook). `empty_cameras=1` tells the model the 3rd slot is intentionally masked rather than erroring on a feature-count mismatch, and `rename_map` maps our camera names onto the `camera1`/`camera2` keys the checkpoint expects.

## ⚙️ Requirements
- Your dataset already on the Hugging Face Hub (default here: `hungdo2401/so101_baseline`, 50 episodes)
- Optional: a [wandb](https://wandb.ai/) account for training visualization
- A GPU runtime: **Runtime > Change runtime type > T4 GPU** (or better, if you have Colab Pro access to an A100)

## ⏱️ Expected training time
LeRobot's own guidance: 20,000 steps ≈ 4-5 hours on an A100 at `batch_size=64`. A free-tier Colab **T4** has much less VRAM than an A100, so this notebook defaults to a smaller `batch_size=8` to fit -- see the training cell for how to tune it up if you have more GPU memory.

## Install LeRobot
Clones `lerobot`, installs FFmpeg, and installs the package in editable mode with the `train`, `dataset`, and `smolvla` extras.

In [ ]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install -y ffmpeg
!cd lerobot && pip install -e ".[train, dataset, smolvla]"

## Weights & Biases login (optional)
Skip this cell if you don't want experiment tracking. If you do run it, remember to set `--wandb.enable=true` in the training cell below.

In [ ]:
!wandb login

## Hugging Face login
Needed to read your private dataset (if it's private) and to push the trained checkpoint back to the Hub.

1. Run the cell below.
2. Generate a token at https://huggingface.co/settings/tokens if you don't already have one.
3. Select all checkboxes under **Repositories** when creating the token (read + write).
4. Paste the token into the prompt.

In [ ]:
!hf auth login

## Sanity-check the dataset before committing to a long run

Confirms the camera feature names the `--rename_map` below assumes actually match what's on the Hub -- if you recorded under a different `--dataset.repo_id` or the camera keys ever change, you'll see it here instead of discovering it hours into training.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

DATASET_REPO_ID = "hungdo2401/so101_baseline"  # <- change if you used a different --repo-id when recording

meta = LeRobotDatasetMetadata(DATASET_REPO_ID)
print(f"episodes: {meta.total_episodes}")
print(f"features: {list(meta.features.keys())}")

expected_cameras = {"observation.images.front", "observation.images.wrist"}
found_cameras = {k for k in meta.features if k.startswith("observation.images.")}
assert found_cameras == expected_cameras, (
    f"Camera keys don't match what --rename_map below assumes. "
    f"Found {found_cameras}, expected {expected_cameras}. Update the training cell's "
    f"--rename_map to match before running it."
)
print("Camera keys match what the training cell's --rename_map expects.")

## Start training SmolVLA on the so101_baseline dataset

Adjust before running:

1. **`--dataset.repo_id`** -- your dataset (already set to `hungdo2401/so101_baseline`, 50 episodes, `front`+`wrist` cameras).
2. **`--policy.repo_id`** -- where the fine-tuned checkpoint gets pushed on the Hub when training finishes.
3. **`--batch_size=8`** -- sized for a free-tier T4. If you're on an A100 (Colab Pro), you can raise this significantly (LeRobot's own guidance uses 64) -- start low and increase incrementally as long as loading times stay short (same tip as the official docs).
4. **`--steps=20000`** -- LeRobot's standard default. With only 50 episodes you may see diminishing returns earlier; watch the loss curve (wandb, if enabled) and stop early via `--policy.pretrained_path=...` resume if it plateaus, rather than assuming more steps always helps.
5. **`--policy.device=cuda`** -- use `cpu` only as a last resort (very slow for a 450M-parameter model) if no GPU runtime is available.
6. **`--wandb.enable=false`** -- flip to `true` if you ran the wandb login cell above.

In [ ]:
!cd lerobot && lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --policy.repo_id=hungdo2401/smolvla_so101_baseline \
  --policy.push_to_hub=true \
  --policy.empty_cameras=1 \
  --dataset.repo_id=hungdo2401/so101_baseline \
  --rename_map='{"observation.images.front": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}' \
  --batch_size=8 \
  --steps=20000 \
  --output_dir=outputs/train/smolvla_so101_baseline \
  --job_name=smolvla_so101_baseline \
  --policy.device=cuda \
  --wandb.enable=false

## If it underperforms

Could be data quality (50 episodes is LeRobot's own suggested *minimum* starting point, not a guarantee -- the SmolVLA docs note a 25-episode dataset was NOT enough for their own reference task) or it may just need more steps. To continue training from this run's checkpoint instead of starting over:

```
--policy.pretrained_path=hungdo2401/smolvla_so101_baseline
```

in place of `--policy.path=lerobot/smolvla_base` above (keep `--dataset.repo_id`, `--rename_map`, and `--policy.empty_cameras` the same).